# 03 – Nihai Model, Kilitli Test Tahminleri ve Bağımsız Test Değerlendirmesi
Bu defter iki bölümden oluşur:

| Bölüm | Ne yapar | Test etiketi? |
|---|---|---|
| **A** | Nihai konfigürasyonu tüm development üzerinde **3 başlangıç seed'i** ile eğitir. Birincil seed önceden ilan edilir. Etiketsiz `test_predictions_locked.csv` ve SHA-256 manifestosu üretilir. Şekil 3–4 çizilir. | **Okunmaz** |
| ⛔ | Kilitli dosya + kod + konfigürasyon **ayrı bir commit** ile GitHub'a gönderilir | – |
| **B** | Etiketler ancak bundan sonra okunur: metrikler, bootstrap %95 GA, confusion matrix, ROC/PR, seed kararlılığı, hata analizi, Şekil 1, README | Okunur |

Test performansına göre model, eşik veya ön işlemede **hiçbir değişiklik yapılmaz**.

In [ ]:
# === 0. Ortam kurulumu (üç defterde aynı) ===
import os, sys, subprocess, importlib

# Kendi GitHub reponuzun adresi (boş bırakılırsa Drive'daki YSA_Midterm_Project klasörü kullanılır)
GITHUB_REPO_URL = "https://github.com/James007Bond-code/Yapay-sinir-aglar-.git"   # ör. "https://github.com/<kullanici-adi>/ysa-mlp-bank-marketing-suleyman-balsak.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if GITHUB_REPO_URL:
        repo = GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
        PROJECT_DIR = f"/content/drive/MyDrive/{repo}"
        if not os.path.exists(os.path.join(PROJECT_DIR, "src")):
            subprocess.run(["git", "clone", GITHUB_REPO_URL, PROJECT_DIR], check=True)
        else:   # yalnızca kod güncellemelerini çeker; üretilen sonuçlara dokunmaz
            r = subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], capture_output=True, text=True)
            print("git pull:", (r.stdout or r.stderr).strip()[:200])
    else:
        PROJECT_DIR = "/content/drive/MyDrive/YSA_Midterm_Project"
else:  # yerel Jupyter: defter proje kök dizininde çalıştırılır
    PROJECT_DIR = os.getcwd()

os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for d in ["configs", "results", "results/cache", "figures", "data", "models"]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists("src/ysa_utils.py"):
    raise FileNotFoundError(
        f"{PROJECT_DIR}/src/ysa_utils.py bulunamadı!\n"
        "Çözüm 1: GITHUB_REPO_URL'ye kendi repo adresinizi yazın (repo src/ klasörünü içermeli).\n"
        f"Çözüm 2: Drive'da {PROJECT_DIR}/src/ klasörü açıp ysa_utils.py dosyasını oraya yükleyin.")

import src.ysa_utils as U
importlib.reload(U)                      # modül değişirse yeniden yükle
from src.ysa_utils import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Çalışma dizini:", os.getcwd())

---
# BÖLÜM A – Etiketsiz kilitli tahminler

In [ ]:
import json, time, joblib
import torch
from sklearn.linear_model import LogisticRegression

SEEDS = load_json("configs/seeds.json")
FINAL_CFG = load_json("results/final_hyperparameters.json")
THR = FINAL_CFG.get("threshold", 0.50)
PRIMARY_SEED = SEEDS["model_init_seeds"][0]          # önceden ilan edilen birincil seed

dev_df = pd.read_csv("data/dev_data.csv")
X_dev = prepare_features(dev_df)
y_dev = dev_df[TARGET].astype(int)
g_dev = dev_df[GROUP_COL].values

# Test: YALNIZCA kimlik + girdi özellikleri. Etiket sütunu okunduğu anda atılır.
_test = pd.read_csv("data/test_data.csv")
test_ids = _test[ID_COL].values
X_test = prepare_features(_test.drop(columns=[TARGET]))
del _test
assert TARGET not in X_test.columns and list(X_test.columns) == list(X_dev.columns)

num_cols, cat_cols = get_feature_lists(X_dev)
print(f"Dev: {X_dev.shape} | Test (etiketsiz): {X_test.shape}")
print(f"Nihai konfigürasyon: {FINAL_CFG['config_id']} ({FINAL_CFG['selected_search_method']})")
print("  ", config_label(FINAL_CFG))
print(f"Birincil seed: {PRIMARY_SEED} | Kararlılık seed'leri: {SEEDS['model_init_seeds'][1:]} | Eşik: {THR}")

In [ ]:
locked = pd.DataFrame({ID_COL: test_ids})
final_meta = []
T0 = time.time()
for i, s in enumerate(SEEDS["model_init_seeds"]):
    role = "primary" if s == PRIMARY_SEED else f"stability_{i + 1}"
    # Early-stop alt kümesi tüm seed'lerde aynı (es_seed sabit); yalnızca ağırlık başlangıcı değişir
    res = train_mlp(X_dev, y_dev, {"test": X_test}, FINAL_CFG, num_cols, cat_cols,
                    model_seed=s, es_seed=SEEDS["early_stop_seed"], groups=g_dev)
    p = res["probs"]["test"]
    locked[f"prob_mlp_seed{s}"] = p
    locked[f"pred_mlp_seed{s}"] = (p >= THR).astype(int)

    tag = f"{FINAL_CFG['config_id']}_seed{s}"
    torch.save(res["model"].state_dict(), f"models/mlp_{tag}.pt")
    joblib.dump(res["preprocessor"], f"models/preprocessor_{tag}.joblib")
    save_json(get_learned_stats(res["preprocessor"], num_cols, cat_cols), f"models/learned_stats_{tag}.json")
    history_frame(res, seed=s, role=role).to_csv(f"results/final_training_history_seed{s}.csv", index=False)
    final_meta.append({"role": role, "seed": s, "config_id": FINAL_CFG["config_id"],
                       "best_epoch": res["best_epoch"], "stopped_epoch": res["stopped_epoch"],
                       "n_lr_drops": res["n_lr_drops"], "final_lr": res["final_lr"],
                       "best_es_loss": round(res["best_es_loss"], 5), "n_params": res["n_params"],
                       "input_dim": res["input_dim"], "n_fit": res["n_fit"], "n_es": res["n_es"],
                       "elapsed_sec": round(res["elapsed_sec"], 2), "status": res["status"],
                       "warnings": res["warnings"]})
    if s == PRIMARY_SEED:
        PRIMARY = res

# Referans model: Logistic Regression (aynı sabit ön işleme kuralı, tüm dev üzerinde)
pre_lr = build_preprocessor(num_cols, cat_cols, "StandardScaler", False, "keep_category")
logreg = LogisticRegression(max_iter=2000, random_state=SEEDS["logreg_seed"]).fit(pre_lr.fit_transform(X_dev), y_dev)
p_lr = logreg.predict_proba(pre_lr.transform(X_test))[:, 1]
locked["prob_logreg"] = p_lr
locked["pred_logreg"] = (p_lr >= THR).astype(int)
joblib.dump({"preprocessor": pre_lr, "model": logreg}, "models/logreg_reference.joblib")

assert TARGET not in locked.columns
LOCKED_PATH = "results/test_predictions_locked.csv"
locked.to_csv(LOCKED_PATH, index=False)
pd.DataFrame(final_meta).to_csv("results/final_seeds_train_meta.csv", index=False)

manifest = {
    "file": LOCKED_PATH, "sha256": file_sha256(LOCKED_PATH), "n_rows": int(len(locked)),
    "columns": list(locked.columns), "contains_labels": False,
    "primary_model": f"prob_mlp_seed{PRIMARY_SEED}", "primary_seed": PRIMARY_SEED,
    "stability_seeds": SEEDS["model_init_seeds"][1:], "reference_model": "prob_logreg",
    "threshold": THR,
    "secondary_threshold_mlp": FINAL_CFG["secondary_threshold_mlp"],
    "secondary_threshold_logreg": FINAL_CFG["secondary_threshold_logreg"],
    "final_config": FINAL_CFG,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "note": "Test etiketleri okunmadan üretildi. Metrikler yalnızca bu dosyanın commit'inden sonra hesaplanır.",
}
save_json(manifest, "results/locked_predictions_manifest.json")
print(f"Süre: {time.time() - T0:.1f} sn")
display(pd.DataFrame(final_meta))
print("SHA-256:", manifest["sha256"])
locked.head()

### Şekil 4 – Nihai (birincil seed) modelin eğitim / iç early-stop kaybı ve epoch bazlı LR

In [ ]:
h = pd.DataFrame(PRIMARY["history"])
h.to_csv("results/final_model_training_history.csv", index=False)
be = PRIMARY["best_epoch"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(h["epoch"], h["train_loss"], lw=2, label="Eğitim (fit alt kümesi) BCE")
axes[0].plot(h["epoch"], h["es_val_loss"], lw=2, label="İç early-stop doğrulama BCE")
axes[0].axvline(be, color="red", ls="--", label=f"En iyi epoch = {be} (ağırlıklar geri yüklendi)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCEWithLogitsLoss")
axes[0].set_title(f"{FINAL_CFG['config_id']} – seed {PRIMARY_SEED} (durdu: epoch {PRIMARY['stopped_epoch']})")
axes[0].legend(fontsize=8)
axes[1].step(h["epoch"], h["lr"], where="post", lw=2, color="#55A868")
axes[1].set_yscale("log"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Learning rate (log)")
axes[1].set_title(f"ReduceLROnPlateau – LR düşüş sayısı: {PRIMARY['n_lr_drops']}"
                  + ("" if FINAL_CFG.get("use_scheduler", True) else " (scheduler kapalı)"))
plt.tight_layout(); plt.savefig("figures/fig4_loss_and_lr_curve.png", dpi=300); plt.show()

# Üç seed'in eğitim eğrileri (ek şekil)
fig, ax = plt.subplots(figsize=(7, 4))
for s in SEEDS["model_init_seeds"]:
    hs = pd.read_csv(f"results/final_training_history_seed{s}.csv")
    ax.plot(hs["epoch"], hs["es_val_loss"], label=f"seed {s}")
ax.set_xlabel("Epoch"); ax.set_ylabel("İç early-stop BCE"); ax.set_title("Seed'lere göre iç doğrulama kaybı"); ax.legend()
plt.tight_layout(); plt.savefig("figures/fig4b_seed_es_loss.png", dpi=300); plt.show()

### Şekil 3 – MLP mimari şeması ve öğrenilebilir parametre sayısı

In [ ]:
import matplotlib.patches as mpatches

in_dim = PRIMARY["input_dim"]
n_num = len(num_cols)
n_ohe = in_dim - n_num
hidden = list(FINAL_CFG["hidden_layers"])
dims = [in_dim] + hidden + [1]

layer_rows = []
for i in range(len(dims) - 1):
    w, b = dims[i] * dims[i + 1], dims[i + 1]
    layer_rows.append({"Katman": f"Linear {i + 1}" + (" (çıkış)" if i == len(dims) - 2 else ""),
                       "Girdi": dims[i], "Çıktı": dims[i + 1], "Ağırlık": w, "Bias": b, "Toplam": w + b})
layer_tbl = pd.DataFrame(layer_rows)
total = int(layer_tbl["Toplam"].sum())
assert total == PRIMARY["n_params"] == count_mlp_params(in_dim, hidden)
layer_tbl.loc[len(layer_tbl)] = {"Katman": "TOPLAM", "Girdi": "", "Çıktı": "", "Ağırlık": int(layer_tbl["Ağırlık"].sum()),
                                 "Bias": int(layer_tbl["Bias"].sum()), "Toplam": total}
layer_tbl.to_csv("results/final_model_parameter_count.csv", index=False)
display(layer_tbl)

def box(ax, x, y, w, hgt, txt, fc, ec, fs=9):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, hgt, boxstyle="round,pad=0.1", fc=fc, ec=ec, lw=1.8))
    ax.text(x + w / 2, y + hgt / 2, txt, ha="center", va="center", fontsize=fs, fontweight="bold")

n_box = len(dims)
W, G = 2.3, 1.1
fig, ax = plt.subplots(figsize=(n_box * (W + G) + 0.5, 4.3))
ax.set_xlim(0, n_box * (W + G)); ax.set_ylim(0, 4.3); ax.axis("off")
act = FINAL_CFG["activation"]
for i, d in enumerate(dims):
    x = i * (W + G) + 0.2
    if i == 0:
        txt, fc, ec = f"Girdi\n{d} özellik\n\n{n_num} sayısal\n({FINAL_CFG['scaler']})\n+ {n_ohe} one-hot", "#DCE6F2", "#2B5B84"
    elif i < len(dims) - 1:
        r = layer_rows[i - 1]
        txt = (f"Gizli katman {i}\nLinear {dims[i - 1]}→{d}\n+ {act}"
               + (f" + Dropout {FINAL_CFG['dropout']}" if FINAL_CFG["dropout"] > 0 else "")
               + f"\n\n{r['Ağırlık']:,} ağırlık\n+ {r['Bias']} bias")
        fc, ec = "#FFF2CC", "#BF9000"
    else:
        r = layer_rows[-1]
        txt = (f"Çıkış\nLinear {dims[-2]}→1 (logit)\n\nBCEWithLogitsLoss\nP(y=1|x)=σ(logit)\n"
               f"{r['Ağırlık']} + {r['Bias']} param.")
        fc, ec = "#FCE4D6", "#C55A11"
    box(ax, x, 1.0, W, 2.6, txt, fc, ec, fs=8.5)
    if i < len(dims) - 1:
        ax.annotate("", xy=(x + W + G - 0.18, 2.3), xytext=(x + W + 0.18, 2.3),
                    arrowprops=dict(arrowstyle="-|>", lw=2.2, color="#222222"))
ax.text(n_box * (W + G) / 2, 0.35,
        f"Toplam öğrenilebilir parametre = {total:,}  |  {FINAL_CFG['optimizer']} (lr={FINAL_CFG['lr']:g}, "
        f"wd={FINAL_CFG['weight_decay']:g}), batch={FINAL_CFG['batch_size']}, ReduceLROnPlateau + early stopping (patience=15)",
        ha="center", fontsize=9, style="italic")
plt.title(f"Nihai MLP ({FINAL_CFG['config_id']}) mimarisi", fontweight="bold")
plt.tight_layout(); plt.savefig("figures/fig3_mlp_architecture.png", dpi=300); plt.show()

---
## ⛔ DUR – Kilitli tahminleri GitHub'a commit'leyin
Bölüm B'ye geçmeden önce aşağıdaki dosyaları **ayrı bir commit** olarak gönderin (Bölüm 12.5):

```bash
git add src/ configs/ 01_eda.ipynb 02_nested_cv.ipynb 03_final_test.ipynb \
        results/final_hyperparameters.json results/test_predictions_locked.csv \
        results/locked_predictions_manifest.json results/final_seeds_train_meta.csv models/
git commit -m "Kilitli test tahminleri (etiketsiz) – nihai model ve kod sürümü"
git push
```
Commit hash'ini aşağıdaki hücreye yazın ve `LOCK_COMMITTED = True` yapın. Kamuya açık veri setinde bu işlem kör test garantisi değil, **zaman sıralı bir deney kaydıdır** (Bölüm 15). Bu durum raporda belirtilmelidir.

---
# BÖLÜM B – Test etiketleriyle değerlendirme (yalnızca commit'ten sonra)
> Oturum yeniden başladıysa önce en üstteki **0. Ortam kurulumu** hücresini çalıştırın.

In [ ]:
LOCK_COMMITTED = False        # ← kilitli dosyayı commit'ledikten sonra True yapın
LOCK_COMMIT_HASH = ""         # ← ör. "3f2a9c1"

assert LOCK_COMMITTED, "Önce test_predictions_locked.csv dosyasını GitHub'a commit'leyin (yukarıdaki ⛔ adım)."

import json, time
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix
from sklearn.calibration import calibration_curve

SEEDS = load_json("configs/seeds.json")
FINAL_CFG = load_json("results/final_hyperparameters.json")
manifest = load_json("results/locked_predictions_manifest.json")
LOCKED_PATH = manifest["file"]
assert file_sha256(LOCKED_PATH) == manifest["sha256"], "Kilitli tahmin dosyası commit'ten sonra DEĞİŞMİŞ!"
manifest["commit_hash"] = LOCK_COMMIT_HASH
manifest["labels_opened_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
save_json(manifest, "results/locked_predictions_manifest.json")

THR = manifest["threshold"]
PRIMARY_SEED = manifest["primary_seed"]
locked = pd.read_csv(LOCKED_PATH)

# Etiketler ŞİMDİ okunur
test_full = pd.read_csv("data/test_data.csv")
ev = locked.merge(test_full[[ID_COL, TARGET]], on=ID_COL, how="left", validate="one_to_one")
assert ev[TARGET].notna().all()
y_test = ev[TARGET].astype(int).values
print(f"Test n = {len(y_test)} | pozitif = {y_test.sum()} (%{100 * y_test.mean():.2f}) | eşik = {THR}")
print("Kilitli dosya doğrulandı, SHA-256:", manifest["sha256"][:16], "…")

### Tablo 6 – Bağımsız test metrikleri ve 1.000 tekrarlı bootstrap %95 GA

In [ ]:
P_MLP = ev[f"prob_mlp_seed{PRIMARY_SEED}"].values
P_LR = ev["prob_logreg"].values
P_DUMMY = np.zeros_like(P_MLP)          # çoğunluk sınıfı referansı (eğitim gerektirmez)

models = {f"MLP {FINAL_CFG['config_id']} (birincil seed {PRIMARY_SEED})": P_MLP,
          "Logistic Regression": P_LR, "Dummy (çoğunluk sınıfı)": P_DUMMY}
test_rows = []
for name, p in models.items():
    m = compute_metrics(y_test, p, THR)
    m["Model"] = name
    test_rows.append(m)
# İkincil sonuç: kilitlemeden ÖNCE development OOF ile seçilmiş eşikler (ana sonuç değildir)
for name, p, t in [(list(models)[0], P_MLP, manifest["secondary_threshold_mlp"]),
                   ("Logistic Regression", P_LR, manifest["secondary_threshold_logreg"])]:
    m = compute_metrics(y_test, p, t)
    m["Model"] = f"{name} – ikincil eşik {t:.3f}"
    test_rows.append(m)
test_metrics = pd.DataFrame(test_rows)[["Model"] + METRIC_COLS + ["TP", "FP", "TN", "FN", "threshold"]]

ci_rows, rejected = [], {}
for name, p in list(models.items())[:2]:
    ci, rej = bootstrap_ci(y_test, p, n_boot=1000, seed=SEEDS["bootstrap_seed"], threshold=THR)
    rejected[name] = rej
    for met, d in ci.items():
        ci_rows.append({"Model": name, "Metrik": met, "Nokta tahmini": d["point"],
                        "GA alt (%2.5)": d["ci_low"], "GA üst (%97.5)": d["ci_high"],
                        "Bootstrap SD": d["boot_sd"], "n_boot": 1000, "reddedilen (tek sınıf)": rej})
ci_tbl = pd.DataFrame(ci_rows)

test_metrics.to_csv("results/test_metrics.csv", index=False)
ci_tbl.to_csv("results/test_bootstrap_ci.csv", index=False)
display(test_metrics.round(4))
display(ci_tbl.round(4))
print(f"Pozitif oranı (rastgele sıralayıcının beklenen AP'si): {y_test.mean():.4f}")
print("Ana sonuç 0.50 eşiğindeki birincil MLP satırıdır. 'ikincil eşik' satırları önceden kaydedilmiş ek analizdir.")

### Tablo 7 – Üç başlangıç seed'inin test performansı (betimsel Mean ± SD)

In [ ]:
seed_rows = []
for s in SEEDS["model_init_seeds"]:
    m = compute_metrics(y_test, ev[f"prob_mlp_seed{s}"].values, THR)
    m.update({"seed": s, "role": "primary" if s == PRIMARY_SEED else "stability"})
    seed_rows.append(m)
seed_df = pd.DataFrame(seed_rows)
summary = {"seed": "Mean ± SD", "role": "n = 3"}
for met in METRIC_COLS:
    summary[met] = f"{seed_df[met].mean():.4f} ± {seed_df[met].std(ddof=1):.4f}"
seed_out = pd.concat([seed_df[["seed", "role"] + METRIC_COLS].round(4).astype(object),
                      pd.DataFrame([summary])], ignore_index=True)
seed_out.to_csv("results/seed_stability_results.csv", index=False)
display(seed_out)
print("Ana sonuç önceden ilan edilen birincil seed'dir; en yüksek test skoru seçilmemiştir.")

### Şekil 6 – Confusion matrix (MLP ve Logistic Regression)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, (name, p) in zip(axes, list(models.items())[:2]):
    cm = confusion_matrix(y_test, (p >= THR).astype(int), labels=[0, 1])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    annot = np.array([[f"{cm[i, j]}\n(%{pct[i, j]:.1f})" for j in range(2)] for i in range(2)])
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Tahmin: no (0)", "Tahmin: yes (1)"], yticklabels=["Gerçek: no (0)", "Gerçek: yes (1)"])
    ax.set_title(f"{name}\n(eşik = {THR}; satır yüzdesi)", fontsize=10)
plt.tight_layout(); plt.savefig("figures/fig6_confusion_matrices.png", dpi=300); plt.show()

### Şekil 7 – ROC ve Precision–Recall eğrileri

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ["#C44E52", "#4C72B0"]
for (name, p), c in zip(list(models.items())[:2], colors):
    fpr, tpr, _ = roc_curve(y_test, p)
    m = compute_metrics(y_test, p, THR)
    axes[0].plot(fpr, tpr, lw=2, color=c, label=f"{name}  ROC-AUC = {m['ROC_AUC']:.3f}")
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].step(rec, prec, where="post", lw=2, color=c, label=f"{name}  AP = {m['AP']:.3f}")
    # 0.50 eşiğindeki çalışma noktası
    axes[0].scatter(1 - m["Specificity"], m["Sensitivity"], color=c, s=50, zorder=5, edgecolor="black")
    axes[1].scatter(m["Sensitivity"], m["Precision"], color=c, s=50, zorder=5, edgecolor="black")
axes[0].plot([0, 1], [0, 1], "k:", label="Rastgele (AUC = 0.5)")
axes[0].set_xlabel("1 − Specificity (FPR)"); axes[0].set_ylabel("Sensitivity (TPR)"); axes[0].set_title("ROC – bağımsız test")
axes[1].axhline(y_test.mean(), color="gray", ls=":", label=f"Pozitif oranı = {y_test.mean():.3f}")
axes[1].set_xlabel("Recall (Sensitivity)"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision–Recall – bağımsız test")
for ax in axes:
    ax.legend(fontsize=8, loc="best")
plt.figtext(0.5, -0.02, "Noktalar: 0.50 eşiğindeki çalışma noktası. AP, average_precision_score ile hesaplanmıştır "
            "(trapez PR-AUC değildir).", ha="center", fontsize=8)
plt.tight_layout(); plt.savefig("figures/fig7_roc_pr_curves.png", dpi=300, bbox_inches="tight"); plt.show()

### Hata analizi – False Positive / False Negative (Bölüm 13.2)

In [ ]:
pred = (P_MLP >= THR).astype(int)
grp = np.select([(y_test == 1) & (pred == 1), (y_test == 0) & (pred == 1),
                 (y_test == 1) & (pred == 0), (y_test == 0) & (pred == 0)], ["TP", "FP", "FN", "TN"], default="?")
ea = test_full.set_index(ID_COL).loc[ev[ID_COL]].reset_index()
ea["group"], ea["prob"] = grp, P_MLP
ea["prev_contacted"] = (ea["pdays"] != PDAYS_NO_CONTACT).astype(int)

err_tbl = ea.groupby("group").agg(
    n=("prob", "size"), ort_olasilik=("prob", "mean"), medyan_olasilik=("prob", "median"),
    yas_medyan=("age", "median"), campaign_medyan=("campaign", "median"),
    euribor3m_medyan=("euribor3m", "median"), nr_employed_medyan=("nr.employed", "median"),
    onceki_temas_pct=("prev_contacted", lambda x: 100 * x.mean()),
    poutcome_success_pct=("poutcome", lambda x: 100 * (x == "success").mean()),
    contact_cellular_pct=("contact", lambda x: 100 * (x == "cellular").mean()),
).reindex(["TP", "FN", "FP", "TN"]).round(3)
err_tbl.to_csv("results/error_analysis_groups.csv")
display(err_tbl)

fn = ea[ea.group == "FN"]
near = int(((fn["prob"] >= 0.30) & (fn["prob"] < THR)).sum())
print(f"FN = {len(fn)} pozitifin {len(fn) - near} tanesi olasılık < 0.30 (model bu müşterileri net biçimde 'no' görüyor); "
      f"{near} tanesi eşiğe yakın (0.30–0.50).")
print(f"FP = {(grp == 'FP').sum()} | Pozitif sınıfın %{100 * len(fn) / max(1, y_test.sum()):.1f}'i kaçırıldı (FN).")

top_month = (ea.groupby(["group", "month"]).size() / ea.groupby("group").size()).unstack(fill_value=0).round(3) * 100
top_month.reindex(["TP", "FN", "FP", "TN"]).to_csv("results/error_analysis_month_pct.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for lab, c in [(0, "#4C72B0"), (1, "#DD8452")]:
    sns.histplot(P_MLP[y_test == lab], bins=30, stat="density", alpha=0.55, color=c, ax=axes[0],
                 label=f"Gerçek {'yes' if lab else 'no'} (n={int((y_test == lab).sum())})")
axes[0].axvline(THR, color="red", ls="--", label=f"Eşik = {THR}")
axes[0].set_xlabel("MLP tahmini P(y=1)"); axes[0].set_title("Sınıfa göre tahmin olasılığı dağılımı"); axes[0].legend(fontsize=8)

frac_pos, mean_pred = calibration_curve(y_test, P_MLP, n_bins=10, strategy="quantile")
frac_lr, mean_lr = calibration_curve(y_test, P_LR, n_bins=10, strategy="quantile")
axes[1].plot(mean_pred, frac_pos, "o-", color="#C44E52", label="MLP")
axes[1].plot(mean_lr, frac_lr, "s-", color="#4C72B0", label="LogReg")
axes[1].plot([0, 1], [0, 1], "k:", label="Mükemmel kalibrasyon")
lim = max(mean_pred.max(), frac_pos.max(), mean_lr.max()) * 1.1
axes[1].set_xlim(0, lim); axes[1].set_ylim(0, lim)
axes[1].set_xlabel("Ortalama tahmin olasılığı"); axes[1].set_ylabel("Gözlenen pozitif oranı")
axes[1].set_title("Kalibrasyon eğrisi (opsiyonel, 10 quantile bin)"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/fig8_error_analysis_calibration.png", dpi=300); plt.show()

### Dış CV – development iç CV – bağımsız test karşılaştırması (Bölüm 13.3)

In [ ]:
outer_sum = pd.read_csv("results/outer_cv_summary_mean_sd.csv").set_index("method")
rows = [
    {"Aşama": "Tam development iç CV (nihai seçim)", "Model": FINAL_CFG["config_id"],
     "AP": f"{FINAL_CFG['full_dev_mean_AP']:.4f} ± {FINAL_CFG['full_dev_sd_AP']:.4f}", "ROC_AUC": "–",
     "Not": "Seçim için kullanıldı → iyimser yanlı olabilir"},
]
for mth in ["MLP_Manual", "MLP_Random", "Logistic_Regression", "Dummy_MostFrequent"]:
    if mth in outer_sum.index:
        rows.append({"Aşama": "Dış CV (n = 3 kat)", "Model": mth, "AP": outer_sum.loc[mth, "AP"],
                     "ROC_AUC": outer_sum.loc[mth, "ROC_AUC"], "Not": "Arama prosedürünün genelleme tahmini"})
tm = test_metrics.set_index("Model")
for name in list(models)[:2]:
    rows.append({"Aşama": "Bağımsız test (n = %d)" % len(y_test), "Model": name,
                 "AP": f"{tm.loc[name, 'AP']:.4f}", "ROC_AUC": f"{tm.loc[name, 'ROC_AUC']:.4f}", "Not": "Tek seferlik ölçüm"})
rows.append({"Aşama": "Bağımsız test – 3 seed", "Model": f"MLP {FINAL_CFG['config_id']}",
             "AP": seed_out.iloc[-1]["AP"], "ROC_AUC": seed_out.iloc[-1]["ROC_AUC"], "Not": "Betimsel kararlılık"})
comp = pd.DataFrame(rows)
comp.to_csv("results/cv_vs_test_comparison.csv", index=False)
display(comp)

### Şekil 1 – Veri bölünmesi ve nested CV akış şeması (değerler dosyalardan okunur)

In [ ]:
import matplotlib.patches as mpatches
split_sum = pd.read_csv("results/dataset_split_summary.csv").set_index("Küme")
n_all, n_dev, n_test = (int(split_sum.loc[k, "n"]) for k in ["Tüm veri", "Development (%80)", "Independent Test (%20)"])
ci_mlp = ci_tbl[(ci_tbl.Model == list(models)[0])].set_index("Metrik")

def box(ax, x, y, w, h, txt, fc, ec, fs=8.8):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12", fc=fc, ec=ec, lw=1.8))
    ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=fs, fontweight="bold")

fig, ax = plt.subplots(figsize=(13, 7.2))
ax.set_xlim(0, 13); ax.set_ylim(0, 7.4); ax.axis("off")
box(ax, 3.5, 6.4, 6.0, 0.75, f"UCI Bank Marketing – bank-additional.csv (n = {n_all:,})\n'duration' çıkarıldı → 19 girdi + y | pdays=999 → prev_contacted",
    "#DCE6F2", "#2B5B84")
box(ax, 0.4, 4.95, 6.0, 0.85, f"Development (%80, n = {n_dev:,})\nEDA + 3×3 nested CV + nihai eğitim",
    "#E2EFDA", "#385723")
box(ax, 7.0, 4.95, 5.6, 0.85, f"Bağımsız Test (%20, n = {n_test:,})\nStratified, seed = {SEEDS['split_seed']} — kilitli",
    "#FCE4D6", "#C55A11")
box(ax, 0.4, 3.2, 6.0, 1.15,
    f"Outer StratifiedKFold(3, seed={SEEDS['outer_cv_seed']})\n└ Inner StratifiedKFold(3, seed={SEEDS['inner_cv_seed']}+k)\n"
    "   6 manuel (M1–M6) + 6 otomatik (R1–R6) → iç CV AP ile seçim\n108 iç eğitim + 6 refit + Dummy / LogReg referansları",
    "#FFF2CC", "#BF9000", fs=8.3)
box(ax, 7.0, 3.2, 5.6, 1.15,
    "Kat-içi ön işleme (her eğitimde)\nfit %85 → imputer → P1–P99 clip → scaler | one-hot\n"
    "early-stop %15 → BCE ile ES + ReduceLROnPlateau\nvalidation / outer-eval / test → yalnızca transform",
    "#EDEDED", "#595959", fs=8.3)
box(ax, 0.4, 1.65, 6.0, 0.95,
    f"Tam development iç CV (3 kat × 12 aday = 36 eğitim)\nNihai: {FINAL_CFG['config_id']} ({FINAL_CFG['selected_search_method']}) "
    f"– iç AP = {FINAL_CFG['full_dev_mean_AP']:.4f} ± {FINAL_CFG['full_dev_sd_AP']:.4f}",
    "#D9E1F2", "#2F5597", fs=8.5)
box(ax, 7.0, 1.65, 5.6, 0.95,
    f"3 seed ile nihai eğitim {SEEDS['model_init_seeds']}\n→ etiketsiz test_predictions_locked.csv → ayrı GitHub commit",
    "#E2EFDA", "#385723", fs=8.5)
box(ax, 2.2, 0.2, 8.6, 0.85,
    f"Bağımsız test (birincil seed {PRIMARY_SEED}): AP = {ci_mlp.loc['AP', 'Nokta tahmini']:.4f} "
    f"[{ci_mlp.loc['AP', 'GA alt (%2.5)']:.4f}, {ci_mlp.loc['AP', 'GA üst (%97.5)']:.4f}] | "
    f"ROC-AUC = {ci_mlp.loc['ROC_AUC', 'Nokta tahmini']:.4f} | F1 = {ci_mlp.loc['F1', 'Nokta tahmini']:.4f}\n"
    "1.000 tekrarlı bootstrap %95 GA, eşik = 0.50",
    "#F4CCCC", "#990000", fs=8.8)
kw = dict(arrowstyle="->", lw=1.6, color="#333333")
for (x1, y1), (x2, y2) in [((5.5, 6.38), (3.4, 5.85)), ((7.5, 6.38), (9.8, 5.85)), ((3.4, 4.93), (3.4, 4.4)),
                           ((6.45, 3.78), (6.95, 3.78)), ((3.4, 3.18), (3.4, 2.65)), ((6.45, 2.12), (6.95, 2.12)),
                           ((9.8, 1.63), (8.0, 1.1))]:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=kw)
ax.annotate("", xy=(9.8, 2.65), xytext=(9.8, 4.93), arrowprops=dict(arrowstyle="->", lw=1.5, ls="--", color="#C55A11"))
ax.text(10.0, 2.95, "yalnızca\ntransform", fontsize=7.5, color="#C55A11")
plt.title("Deneysel tasarım: veri bölünmesi, sızıntısız ön işleme ve nested CV", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.savefig("figures/fig1_nested_cv_flow.png", dpi=300); plt.show()

### `requirements.txt`, `README.md` ve `.gitignore` (değerler dosyalardan okunur)

In [ ]:
sysinfo = load_json("results/system_info.json")
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join([
        f"numpy=={sysinfo['numpy_version']}", f"pandas=={sysinfo['pandas_version']}",
        f"scikit-learn=={sysinfo['sklearn_version']}", f"torch=={sysinfo['torch_version'].split('+')[0]}",
        f"matplotlib=={sysinfo['matplotlib_version']}", f"seaborn=={sysinfo['seaborn_version']}", "joblib",
    ]) + "\n")

with open(".gitignore", "w", encoding="utf-8") as f:
    f.write("data/\nresults/cache/\n__pycache__/\n*.pyc\n.ipynb_checkpoints/\n")

seed_last = seed_out.iloc[-1]
PROJECT = load_json("configs/project.json")
AUTHORS = ", ".join(f"{a['name']} ({a['student_id']})" for a in PROJECT["authors"])
INSTR = f" · **Ders sorumlusu:** {PROJECT['instructor']}" if PROJECT.get("instructor") else ""
HISTORY = (f"\n## Deney geçmişi\n{PROJECT['experiment_history_note']}\n" if PROJECT.get("experiment_history_note") else "")
readme = f"""# Bank Marketing – MLP ile Sınıflandırma, Hiperparametre Optimizasyonu ve Nested CV

**{'Hazırlayanlar' if len(PROJECT['authors']) > 1 else 'Hazırlayan'}:** {AUTHORS} · **Ders:** {PROJECT['course']}{INSTR}
**Temel seed:** `{SEEDS['base_seed']}` (= {SEEDS['student_last5']} + 2026; seed kaynağı: {PROJECT['seed_owner']}) – tüm alt seed'ler `configs/seeds.json` içinde
{('> **Seed notu:** ' + PROJECT['seed_note']) if PROJECT.get('seed_note') else ''}

## Veri
UCI Machine Learning Repository – Bank Marketing, yalnızca `bank-additional.csv` (4.119 gözlem).
Moro, S., Rita, P. ve Cortez, P. (2014). https://doi.org/10.24432/C5K306
Ham veri repoya eklenmemiştir; `01_eda.ipynb` resmî bağlantıdan otomatik indirir.
`duration` değişkeni görüşme öncesinde bilinmediği için modellerden çıkarılmıştır.

## Kurulum ve çalıştırma
```bash
git clone <repo-url>  &&  cd <repo>
pip install -r requirements.txt
```
Google Colab'da repo `MyDrive/YSA_Midterm_Project` olarak bulunmalıdır (defterlerin ilk hücresi bu dizini kullanır).
Defterleri **sırasıyla** baştan sona çalıştırın:

1. `01_eda.ipynb` – seed'ler, indirme, kopya kontrolü, %80/%20 stratified ayrım, development-only EDA
2. `02_nested_cv.ipynb` – M1–M6 / R1–R6 adayları, 3×3 nested CV, referans modeller, tam-dev nihai seçim
3. `03_final_test.ipynb` – **Bölüm A:** 3 seed ile nihai eğitim + etiketsiz kilitli tahmin → *ayrı commit* → **Bölüm B:** test metrikleri, bootstrap GA, grafikler

Ortak kod: `src/ysa_utils.py` (ön işleme, MLP, eğitim döngüsü, metrikler, adaylar).

## Klasörler
- `configs/` – seed'ler, arama uzayı, 6 manuel + 6 otomatik aday, eğitim ayarları
- `results/` – bölünme, iç/dış CV sonuçları, eğitim logları, nihai hiperparametreler, kilitli tahminler, test metrikleri
- `figures/` – raporda kullanılan şekiller (Şekil 1–8)
- `models/` – nihai model ağırlıkları, fit edilmiş ön işlemciler ve öğrenilmiş istatistikler
- `report.pdf` – teknik rapor

## Sistem
{sysinfo['environment']} · {sysinfo['os']} · Python {sysinfo['python_version']} · PyTorch {sysinfo['torch_version']} ·
scikit-learn {sysinfo['sklearn_version']} · cihaz: {sysinfo['training_device']} ({sysinfo['cpu_count']} CPU)

## Temel sonuçlar
- Nihai konfigürasyon: **{FINAL_CFG['config_id']}** ({FINAL_CFG['selected_search_method']}) – {config_label(FINAL_CFG)}
- Tam development iç CV AP: {FINAL_CFG['full_dev_mean_AP']:.4f} ± {FINAL_CFG['full_dev_sd_AP']:.4f}
- Bağımsız test (birincil seed {PRIMARY_SEED}, eşik {THR}):
  AP = {ci_mlp.loc['AP', 'Nokta tahmini']:.4f} [{ci_mlp.loc['AP', 'GA alt (%2.5)']:.4f}, {ci_mlp.loc['AP', 'GA üst (%97.5)']:.4f}],
  ROC-AUC = {ci_mlp.loc['ROC_AUC', 'Nokta tahmini']:.4f} [{ci_mlp.loc['ROC_AUC', 'GA alt (%2.5)']:.4f}, {ci_mlp.loc['ROC_AUC', 'GA üst (%97.5)']:.4f}],
  F1 = {ci_mlp.loc['F1', 'Nokta tahmini']:.4f} [{ci_mlp.loc['F1', 'GA alt (%2.5)']:.4f}, {ci_mlp.loc['F1', 'GA üst (%97.5)']:.4f}]
- 3 seed kararlılığı: AP = {seed_last['AP']}, ROC-AUC = {seed_last['ROC_AUC']}
- Kilitli tahmin commit'i: `{LOCK_COMMIT_HASH or 'BELİRTİLMEDİ'}` · SHA-256: `{manifest['sha256'][:16]}…`

Kamuya açık veri setinde test etiketleri erişilebilir olduğundan kilitli commit kör test garantisi değil, zaman sıralı deney kaydıdır.
{HISTORY}"""
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)
print(readme)

### Üretilen dosyaların kontrolü

In [ ]:
need = ["results/test_predictions_locked.csv", "results/locked_predictions_manifest.json", "results/test_metrics.csv",
        "results/test_bootstrap_ci.csv", "results/seed_stability_results.csv", "results/error_analysis_groups.csv",
        "results/cv_vs_test_comparison.csv", "results/final_model_parameter_count.csv", "README.md", "requirements.txt"]
for f in need:
    print(("✓ " if os.path.exists(f) else "✗ ") + f)
print("Şekiller:", sorted(os.listdir("figures")))